[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongodb-partners/partner-cookbooks/blob/main/advanced-rag/02_fulltext_hybrid_and_reranking.ipynb)

# Advanced RAG on MongoDB Atlas: 2. Full-Text, Hybrid & Reranking

Notebook 1 covered semantic (vector) search. Vector search is great at *meaning* but can miss
exact terms like names, places, or rare keywords. This notebook adds the retrieval techniques that
fix that:

## What you'll learn
1. **Full-text search**: classic keyword (BM25) search with an Atlas Search index.
2. **Hybrid search**: combine vector and full-text results with MongoDB's native `$rankFusion`.
3. **Reranking**: reorder candidates with a cross-encoder for sharper relevance.
4. Compare all four retrievers on the same query and see where each wins.

## Prerequisites
Finish `01_atlas_setup_and_vector_search.ipynb` first. It creates the `movies_vector_index`
this notebook reuses. Same credentials as before.


## Setup

In [1]:
%%capture
%pip install -q \
    langchain==1.3.12 langchain-mongodb==0.11.0 langchain-anthropic==1.4.8 \
    pymongo==4.17.0 python-dotenv==1.2.2 requests==2.34.2

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from utils import get_secret, connect, AtlasEmbeddings

MONGODB_URI  = get_secret("MONGODB_URI",  "MongoDB Atlas connection string: ")
ATLAS_AI_KEY = get_secret("MDB_ATLAS_API_KEY", "Atlas API key (for embeddings/rerank): ")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "voyage-3-large")
EMBEDDING_DIMS  = int(os.getenv("EMBEDDING_DIMENSIONS", "2048"))
DB_NAME         = os.getenv("MONGODB_DB", "sample_mflix")

client = connect(MONGODB_URI)
movies = client[DB_NAME]["embedded_movies"]
embeddings = AtlasEmbeddings(api_key=ATLAS_AI_KEY, model=EMBEDDING_MODEL, dimensions=EMBEDDING_DIMS)

VECTOR_INDEX = "movies_vector_index"      # created in notebook 1
FULLTEXT_INDEX = "movies_fulltext_index"  # created below
print("Ready.")

Ready.


## 1. Full-text (keyword) search

A **full-text search index** lets Atlas rank documents by keyword relevance (the BM25 algorithm),
just like a search engine. We index the `plot` field.


In [3]:
from langchain_mongodb.index import create_fulltext_search_index

if FULLTEXT_INDEX not in {ix["name"] for ix in movies.list_search_indexes()}:
    print("Creating full-text index, waiting until queryable...")
    create_fulltext_search_index(
        collection=movies, index_name=FULLTEXT_INDEX,
        field="plot", wait_until_complete=420,
    )
    print("Index is ready.")
else:
    print("Index already exists.")

Index already exists.


In [4]:
from langchain_mongodb.retrievers import MongoDBAtlasFullTextSearchRetriever

fulltext_retriever = MongoDBAtlasFullTextSearchRetriever(
    collection=movies, search_index_name=FULLTEXT_INDEX,
    search_field="plot", k=5,
)

print("Full-text results for 'shark attack in the ocean':")
for d in fulltext_retriever.invoke("shark attack in the ocean"):
    print(f"  - {d.metadata['title']}")

Full-text results for 'shark attack in the ocean':
  - Deep Rising
  - How I Ended This Summer
  - The Poseidon Adventure
  - Ewoks: The Battle for Endor
  - Rebellion


Full-text search matches the *words* you typed. That's exactly right when the query contains
specific terms, but it can't tell that "a great white terrorizes swimmers" is about the same thing
as "shark attack." Vector search handles that; full-text handles exact terms. **Hybrid search
gives you both.**

## 2. Hybrid search with `$rankFusion`

MongoDB 8.0 introduced `$rankFusion`, which runs a vector search and a full-text search and fuses
their rankings server-side using Reciprocal Rank Fusion, in one query with no client-side merging.
`MongoDBAtlasHybridSearchRetriever` wraps it: give it the vector store from notebook 1 and the
full-text index name.


In [5]:
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_mongodb.retrievers import MongoDBAtlasHybridSearchRetriever

vector_store = MongoDBAtlasVectorSearch(
    collection=movies, embedding=embeddings, index_name=VECTOR_INDEX,
    text_key="plot", embedding_key="plot_embedding_voyage_3_large",
    relevance_score_fn="cosine",
)

hybrid_retriever = MongoDBAtlasHybridSearchRetriever(
    vectorstore=vector_store, search_index_name=FULLTEXT_INDEX, k=5,
)

print("Hybrid results for 'shark attack in the ocean':")
for d in hybrid_retriever.invoke("shark attack in the ocean"):
    print(f"  - {d.metadata['title']}")

Hybrid results for 'shark attack in the ocean':


  - Deep Rising
  - Deep Blue Sea
  - How I Ended This Summer
  - Deep Blue Sea
  - The Poseidon Adventure


The hybrid results are noticeably more on-topic: the semantic side pulls in the right *kind* of
movie while the keyword side keeps exact matches. You can tune the balance with `vector_weight`
and `fulltext_weight` if one signal should count more than the other.

## 3. Reranking for sharper relevance

Fusion is fast but coarse. A **reranker** is a model that scores each candidate against the query
directly, so it can reorder a shortlist much more precisely. The pattern: retrieve a wider pool
(say 10), rerank, keep the best few. `atlas_rerank` (in `utils.py`) calls MongoDB's hosted
`rerank-2.5` model.


In [6]:
from utils import atlas_rerank

query = "shark attack in the ocean"
pool = hybrid_retriever.invoke(query)  # returns k=5 candidates; raise k for a bigger pool
titles_by_plot = {d.page_content: d.metadata["title"] for d in pool}

print("Before rerank:")
for d in pool:
    print(f"  - {d.metadata['title']}")

reranked_plots = atlas_rerank(query, [d.page_content for d in pool],
                              api_key=ATLAS_AI_KEY, top_n=3)
print("\nAfter rerank (top 3):")
for plot in reranked_plots:
    print(f"  - {titles_by_plot[plot]}")

Before rerank:
  - Deep Rising
  - Deep Blue Sea
  - How I Ended This Summer
  - Deep Blue Sea
  - The Poseidon Adventure



After rerank (top 3):
  - Deep Blue Sea
  - Deep Blue Sea
  - Deep Rising


## 4. Side-by-side comparison

The same query through each retriever makes the trade-offs concrete.


In [7]:
query = "a giant creature from the deep sea attacks a ship"

def titles(docs):
    return [d.metadata["title"] for d in docs]

vec = titles(vector_store.similarity_search(query, k=5))
ft  = titles(fulltext_retriever.invoke(query))
hyb = titles(hybrid_retriever.invoke(query))

print(f"{'Vector':<28}{'Full-text':<28}{'Hybrid':<28}")
print("-" * 84)
for i in range(5):
    row = [lst[i] if i < len(lst) else "" for lst in (vec, ft, hyb)]
    print(f"{row[0][:26]:<28}{row[1][:26]:<28}{row[2][:26]:<28}")

Vector                      Full-text                   Hybrid                      
------------------------------------------------------------------------------------
Deep Rising                 The Water Horse             The Water Horse             
Pacific Rim                 Sky Captain and the World   Deep Rising                 
Dead Sea                    Big Ass Spider!             Pacific Rim                 
20,000 Leagues Under the S  Mermaid Chronicles Part 1:  Sky Captain and the World   
Dagon                       Carny                       Dead Sea                    


## Recap

You built keyword (full-text) search, combined it with vector search via native `$rankFusion`
hybrid retrieval, and sharpened the ranking with a reranker: the core toolkit of modern retrieval.

**Next:** `03_graphlookup_and_auto_embedding.ipynb` explores two more MongoDB capabilities:
traversing relationships with `$graphLookup` (a movie knowledge graph) and letting Atlas generate
embeddings automatically at write time.
